# 6. The command line and the server

`carbon-ocr` is the engine as a program. It reads images and PDFs, writes eight formats, times models and sets, and serves pages over HTTP. Built with CMake (`cmake --preset ninja-release`, then `cmake --build --preset ninja-release`), it lands in `build/<preset>/cli/` beside `carbon-pdf`, the helper that renders PDF pages. `helpers.carbon_ocr_program()` finds it there, on the `PATH`, or where the `CARBON_OCR` environment variable points. This notebook runs it with `subprocess`, as a script or a service would.

In [1]:
import json
import os
import re
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt

import helpers
import sample_pages

CLI = helpers.carbon_ocr_program()
if CLI is None:
    raise SystemExit("carbon-ocr not found: build it with CMake (Examples/README.md) or set CARBON_OCR to its path")
SET = os.path.relpath(helpers.model_set())  # the set as the program's messages will name it
pages = sample_pages.make_all("data")
Path("out").mkdir(exist_ok=True)


def run(*args, timeout=300):
    """carbon-ocr with these arguments: its standard output; its standard error raised if it fails."""
    r = subprocess.run([CLI, *map(str, args)], capture_output=True, text=True, encoding="utf-8", timeout=timeout)
    if r.returncode:
        raise RuntimeError(r.stderr)
    return r.stdout


print(run("version"))
print(run("cpus"))

carbon-ocr 0.1.0 (backends: ort, native)

GenuineIntel: 13th Gen Intel(R) Core(TM) i7-13620H
kernels: scalar, avx2, avx-vnni (avx-vnni in use)
10 physical cores, 16 logical processors, 1 processor group, 1 NUMA node; no CPU quota
10 workers by default, one per physical core, in this order:
  worker  group  node  class  logical processors
       0      0     0      1  4 5
       1      0     0      1  6 7
       2      0     0      1  2 3
       3      0     0      1  8 9
       4      0     0      1  10 11
       5      0     0      1  0 1
       6      0     0      0  12
       7      0     0      0  13
       8      0     0      0  14
       9      0     0      0  15



`cpus` shows what the engine found: the instruction sets its kernels will use, and the physical cores its workers will be pinned to, fast cores first.

## Reading pages: `carbon-ocr ocr`

`ocr` takes images and PDFs, any number of them, and writes plain text by default: each page's lines in reading order, the pages apart by a form feed. It prints a summary of the run on standard error.

In [2]:
print(run("ocr", pages["letter"], "--set", SET))

Notes on the Harbour Survey
Prepared for the town council, 14 March 2026
The survey of the harbour began on the first calm morning of the month. Two boats
took soundings along the channel every fifty metres, and a tide gauge on the north pier
logged the water level every ten minutes. By the end of the week the crews had
recorded 1,284 depths and 1,008 tide readings.
The channel is shallower than the 2019 chart shows. Near the harbour mouth the least
depth found was 4.6 m at low water, against 5.3 m on the chart; the silt has moved
east, toward the old ferry slip. The inner basin has changed little (its depths are within
0.2 m of the chart's).
Small boats can use the channel at any state of the tide. Vessels drawing more than 4
m should enter within two hours of high water until the channel is dredged.
Findings
• The east bank of the channel has moved 12 m toward the fairway since 2019.
• Tide heights at the pier agree with the predictions to within 9 cm.
• The ferry slip needs a new fe

## JSON

`--format json` gives everything the engine knows about each page. That is more than the Python module's dicts hold: each line's words with their boxes and confidences, the lines the gate dropped and why, and the page's `source` (`ocr`, `text layer` or `hybrid`).

In [3]:
doc = json.loads(run("ocr", pages["columns"], pages["table"], "--set", SET, "--format", "json"))
page = doc["pages"][0]
print(list(page))
line = page["lines"][0]
print({k: v for k, v in line.items() if k != "words"})
for word in line["words"]:
    print("   word", word)
print("dropped:", page["dropped"])
table = next(b for b in doc["pages"][1]["blocks"] if b["label"] == "Table")
print("a table's first cells:", table["cells"][:3])

['file', 'width', 'height', 'rotation', 'skew', 'source', 'order', 'second_pass', 'lines', 'pictures', 'dropped', 'blocks', 'ms']
{'box': [140.4, 84.2, 521.6, 122.5], 'text': 'FIELD JOURNAL - AUTUMN 2026', 'confidence': 0.9754, 'det_score': 0.9343}
   word {'box': [150.0, 84.2, 210.3, 122.5], 'text': 'FIELD', 'confidence': 0.9999}
   word {'box': [217.0, 84.2, 331.0, 122.5], 'text': 'JOURNAL', 'confidence': 0.9998}
   word {'box': [337.7, 84.2, 344.4, 122.5], 'text': '-', 'confidence': 0.4366}
   word {'box': [357.8, 84.2, 458.3, 122.5], 'text': 'AUTUMN', 'confidence': 0.9999}
   word {'box': [471.8, 84.2, 521.6, 122.5], 'text': '2026', 'confidence': 1.0}
dropped: [{'box': [942.3, 1998.5, 1456.1, 2049.1], 'text': '__________________', 'confidence': 0.8628, 'why': 'rules'}, {'box': [1502.3, 2005.8, 1523.7, 2048.3], 'text': '___', 'confidence': 0.9079, 'why': 'rules'}]
a table's first cells: [{'row': 0, 'col': 0, 'lines': [4]}, {'row': 0, 'col': 2, 'lines': [3], 'col_span': 2}, {'row': 0

## Every format

| `--format` | what |
|---|---|
| `text` | the lines in reading order; pages apart by a form feed |
| `json` | every page's lines, words, pictures, blocks, cells, dropped lines and timings |
| `hocr` | hOCR 1.2, the format Tesseract writes and OCRmyPDF reads |
| `alto` | ALTO XML 4.4: text blocks with their labels, lines, words with their confidences |
| `page` | PAGE XML, one file a page in `--out-dir` |
| `markdown` | headings, paragraphs, lists, tables (as HTML) and captions, from the order model's blocks |
| `html` | an HTML5 document, a `<section>` a page: headings, paragraphs, lists, tables, figures, running headers and footers |
| `pdf` | a searchable PDF: each page's image with its words as invisible text over it (`--output` needed) |

`--output FILE` writes to a file instead of standard output, and `--text-out FILE` also writes the plain text beside any format.

In [4]:
for fmt in ["hocr", "alto", "markdown", "html"]:
    out = run("ocr", pages["table"], "--set", SET, "--format", fmt)
    print(f"--- {fmt}: {len(out)} characters")
    print(out[:420].rstrip(), "\n")
run("ocr", pages["table"], "--set", SET, "--format", "page", "--out-dir", "out")
print("PAGE XML:", sorted(p.name for p in Path("out").glob("*.xml")))

--- hocr: 17349 characters
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN"
    "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en" lang="en">
 <head>
  <title></title>
  <meta http-equiv="Content-Type" content="text/html;charset=utf-8"/>
  <meta name="ocr-system" content="carbon-ocr 0.1.0"/>
  <meta name="ocr-capabilities" cont 



--- alto: 19022 characters
<?xml version="1.0" encoding="UTF-8"?>
<alto xmlns="http://www.loc.gov/standards/alto/ns-v4#" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.loc.gov/standards/alto/ns-v4# http://www.loc.gov/alto/v4/alto-4-4.xsd">
 <Description>
  <MeasurementUnit>pixel</MeasurementUnit>
  <sourceImageInformation>
   <fileName>data\table.png</fileName>
  </sourceImageInformation>
  <OCRProcessing 



--- markdown: 1102 characters
## Ferry passengers by quarter, 2025

Passenger numbers rose in every quarter on every route. The third quarter, with the summer timetable, carried a third of the year's passengers.

<table>
<tr><td>Route</td><td></td><td colspan="2">Passengers (thousands)</td><td></td><td>Year</td></tr>
<tr><td></td><td>Q1</td><td>Q2</td><td>Q3</td><td>Q4</td><td></td></tr>
<tr><td>Harbour to Northgate</td><td>41.2</td><td>44.8</td> 



--- html: 1326 characters
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<meta name="generator" content="Carbon-OCR 0.1.0">
<title>Carbon-OCR</title>
</head>
<body>
<section class="page" id="page-1">
<h2>Ferry passengers by quarter, 2025</h2>
<p>Passenger numbers rose in every quarter on every route. The third quarter, with the summer timetable, carried a third of the year's passengers.</p>
<table>
<tr><td>Route</td><td></td><td colspan 



PAGE XML: ['table.xml']


## The engine's options on the command line

Most of the options notebook 2 tried have a flag:

| flag | `Engine` option |
|---|---|
| `--workers N`, `--no-pin` | `workers`, `pin=False` |
| `--precision auto\|fp32\|fp16\|int8`, `--backend ort\|native` | `precision`; one backend for every model |
| `--rec NAME` | `rec` |
| `--gate auto\|off` | `gate` |
| `--block-order geometry\|model` | `block_order` |
| `--table-spans on\|off` | `table_spans` |
| `--second-pass auto\|off`, `--second-pass-share F` | the second pass at all, and `second_pass_share` |
| `--single-pixel F` | `single_pixel` |
| `--spaces model\|ink` | `spaces_from_ink` |
| `--as-given` | no quarter turns and no deskew: the page as given |
| `--dpi N\|auto`, `--text-layer auto\|off`, `--hybrid auto\|off` | PDF pages: their resolution, their own text, the pictures of a text page |

`justified`, `det_area`, `det_pitch`, `second_pass_target`, `flip_needs_dark`, `char_confidence`, `fused_rec_head`, `det_stem` and `word_boxes_at_ink` are set from Python or C++.

In [5]:
import carbon_ocr  # for its scoring

truth = sample_pages.truth_text(pages["newspaper"])
for mode in ["auto", "off"]:
    t = time.perf_counter()
    text = run("ocr", pages["newspaper"], "--set", SET, "--second-pass", mode)
    s = time.perf_counter() - t
    e = carbon_ocr.text_errors([truth], [text])[0]
    print(f"--second-pass {mode:4}: {len(text.splitlines())} lines, CER {e[0] / e[1]:.2%}, WER {e[2] / e[3]:.2%}, {s:.2f} s")

--second-pass auto: 945 lines, CER 0.51%, WER 0.08%, 1.10 s


--second-pass off : 855 lines, CER 1.01%, WER 1.41%, 0.93 s


The second pass, which the Python module always takes where a page needs it, can be turned off here. On the newspaper, without it, the CER doubles and many more words are wrong, for about a fifth less time.

## PDFs

`ocr` reads PDFs as well: PDFium renders each page in `carbon-pdf` helper processes, so a PDF that crashes PDFium takes down only a helper. A born-digital page whose own text passes the engine's checks is read from that text instead of by OCR. Its text must be there, readable, visible and inked on the page. `--format pdf` writes a searchable PDF: each page's image, with its words over it as invisible text.

In [6]:
print(run("ocr", pages["letter"], pages["scan"], "--set", SET, "--format", "pdf", "--output", "out/searchable.pdf"))
print(f"out/searchable.pdf: {Path('out/searchable.pdf').stat().st_size / 1e3:.0f} kB")
back = json.loads(run("ocr", "out/searchable.pdf", "--set", SET, "--format", "json"))["pages"][0]
print("read back:", back["source"], "-", back["text_layer"])


out/searchable.pdf: 1249 kB


read back: ocr - 1104 of 1104 characters are invisible


A searchable PDF's words are invisible, as an OCR layer over a scan always is, so the engine does not trust them: it reads the page again by OCR and says why. A born-digital PDF draws its text. Here matplotlib makes one, with its fonts embedded:

In [7]:
plt.rcParams["pdf.fonttype"] = 42  # TrueType fonts in the PDF, with their characters' Unicode
fig = plt.figure(figsize=(8.5, 11))
for i, s in enumerate(["Notes on the Harbour Survey", sample_pages.HARBOUR[0][:62], sample_pages.HARBOUR[1][:60]]):
    fig.text(0.1, 0.9 - 0.04 * i, s, fontsize=20 if i == 0 else 13, family="DejaVu Serif")
fig.savefig("out/born-digital.pdf")
plt.close(fig)
for flags in [[], ["--text-layer", "off"], ["--dpi", "auto"]]:
    page = json.loads(run("ocr", "out/born-digital.pdf", "--set", SET, "--format", "json", *flags))["pages"][0]
    print(f"{' '.join(flags) or 'the defaults':18} source {page['source']!r:13} {page['width']} x {page['height']} px:",
          [line["text"][:30] for line in page["lines"]])

the defaults       source 'text layer'  1700 x 2200 px: ['Notes on the Harbour Survey', 'The survey of the harbour bega', 'The channel is shallower than ']


--text-layer off   source 'ocr'         1700 x 2200 px: ['Notes on the Harbour Survey', 'The survey of the harbour bega', 'The channel is shallower than ']


--dpi auto         source 'text layer'  1700 x 2200 px: ['Notes on the Harbour Survey', 'The survey of the harbour bega', 'The channel is shallower than ']


The page's own text is read in a few milliseconds and needs no models. `--text-layer off` reads it by OCR instead, at the page's rendering (200 dpi by default; `--dpi auto` picks each page's from its small print or its scan, from 200 to 400 dpi).

## Timing and inspecting

| command | what |
|---|---|
| `info MODEL` | a model's inputs and outputs |
| `bench MODEL --shape x=NxCxHxW` | one model timed on fixed inputs: median, p90, fastest |
| `compare MODEL` | a model on the engine's own kernels against ONNX Runtime: how far apart their outputs are |
| `pages SET --workers 1,all` | a whole set timed as the CPU engine runs a page, at each number of workers |

In [8]:
rec = os.path.join(SET, "rec-latin.onnx")
print(run("info", rec))
print(run("bench", rec, "--shape", "x=64x1x32x640", "--runs", "20"))
print(run("compare", rec, "--shape", "x=8x1x32x512"))

..\models\fullerene-3M\rec-latin.onnx on ort
  input   x                        float32  [-1, 1, 32, -1]
  output  logits                   float32  [-1, 800, -1]



ort, 20 runs after 10 warm-up: median 262.792 ms, p90 268.125 ms, min 255.601 ms
  output  logits                   [64, 800, 160]

..\models\fullerene-3M\rec-latin.onnx: native (avx-vnni) against ort
  logits       [8, 800, 128]      largest difference 7.92e-05, largest value 21.6 (3.7e-06 of it)



In [9]:
print(run("pages", SET, "--workers", "1,all", "--pages", "16"))

..\models\fullerene-3M: a page is orient and det, 40 lines in 5 to 40 tasks (rec-latin), then order; 16 pages, all queued
  orient.onnx on ort, det.onnx on ort, rec-latin.onnx on ort, order.onnx on ort
  workers    pages/s  per worker  ms a page (median, p90)   scaling  line tasks  det threads  det strips  to the lines  the lines
  4, 8 and 16 strips give the whole page's maps, byte for byte
        1       5.35        5.35       185.1       190.2       100.0%           5            1           1          47.6      137.5
       10      18.63        1.86       406.1       541.9        34.8%           5            1           1         161.4      244.8



`pages` reads pages at the models' design sizes, 40 lines each. One worker shows a page's latency on one core, and `all` the throughput of every core with every page queued at once.

## The server: `carbon-ocr serve`

`serve` loads the set once and reads pages that clients send over HTTP/1.1, until it is stopped. Every client's pages share the engine's workers: each page goes to the next free worker and is answered as soon as it is read. It takes `ocr`'s model, worker and device options. `--port 0` picks a free port, which its first line on standard error gives.

| request | answer |
|---|---|
| `POST /ocr` | the body a page's image (PNG, JPEG, BMP or PNM); the answer that page in `?format=` (`json` by default, or any of `ocr`'s formats); `X-Carbon-Ms` and `X-Carbon-Lines` give its time in the engine and its lines |
| `GET /health` | `ok`, or 503 once a GPU error has stopped the pipeline |
| `GET /info` | the version, the set, the device, the precision, the workers, the pages in flight and the queue |

In [10]:
server = subprocess.Popen([CLI, "serve", "--set", SET, "--port", "0", "--quiet"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True, encoding="utf-8")
first = server.stderr.readline()
print(first.strip())
URL = re.search(r"http://[\d.]+:\d+", first).group()


def call(path, body=None):
    """A request to the server: (status, headers, body as text)."""
    req = urllib.request.Request(URL + path, data=body, method="POST" if body is not None else "GET")
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return r.status, r.headers, r.read().decode("utf-8")
    except urllib.error.HTTPError as e:
        return e.code, e.headers, e.read().decode("utf-8")


print(call("/health")[2].strip())
print(json.loads(call("/info")[2]))

carbon-ocr 0.1.0 serving ..\models\fullerene-3M on http://127.0.0.1:63001 (the CPU): POST /ocr, GET /health, GET /info; Ctrl+C stops it
ok
{'version': '0.1.0', 'set': '../models/fullerene-3M', 'device': 'cpu', 'precision': 'auto', 'workers': 10, 'in_flight': 20, 'queue': 64}


In [11]:
png = pages["letter"].read_bytes()
status, headers, body = call("/ocr", png)
page = json.loads(body)
print(status, "| X-Carbon-Ms", headers["X-Carbon-Ms"], "| X-Carbon-Lines", headers["X-Carbon-Lines"])
print(page["lines"][0]["text"], page["lines"][0]["words"][0])
print(call("/ocr?format=markdown", pages["table"].read_bytes())[2][:300])
status, _, reason = call("/ocr", b"not an image")
print(status, reason)

200 | X-Carbon-Ms 61.4 | X-Carbon-Lines 21
Notes on the Harbour Survey {'box': [185.6, 162.0, 332.7, 238.4], 'text': 'Notes', 'confidence': 1.0}


## Ferry passengers by quarter, 2025

Passenger numbers rose in every quarter on every route. The third quarter, with the summer timetable, carried a third of the year's passengers.

<table>
<tr><td>Route</td><td></td><td colspan="2">Passengers (thousands)</td><td></td><td>Year</td></tr>
<tr><td></t
415 the body is no image the server reads: PNG, JPEG, BMP or PNM



In [12]:
from concurrent.futures import ThreadPoolExecutor

bodies = [pages[n].read_bytes() for n in ["letter", "columns", "table", "banner", "receipt", "slide"]] * 4
t = time.perf_counter()
with ThreadPoolExecutor(8) as clients:  # eight clients at once
    statuses = list(clients.map(lambda b: call("/ocr?format=text", b)[0], bodies))
s = time.perf_counter() - t
print(f"{len(bodies)} pages from 8 clients in {s:.2f} s: {len(bodies) / s:.1f} pages a second, statuses {set(statuses)}")
server.terminate()
server.wait()
print("the server is stopped")

24 pages from 8 clients in 0.65 s: 37.0 pages a second, statuses {200}
the server is stopped


The server has no TLS and no authentication. To serve beyond the machine, put it behind a proxy that has both. Errors are answered as HTTP statuses, with the reason as the body: 415 for a body that is no image it reads, 422 for an image that does not decode, 413 past `--max-body-mb`, and 503 with `Retry-After: 1` when its queue is full.

## On a GPU

With a build that has CUDA and TensorRT (the [build guide](https://github.com/karanvora2599/Carbon-OCR/blob/main/docs/BUILD.md) gives its presets), the same commands run the batched pipeline on an NVIDIA GPU. TensorRT builds an engine for each model and shape the first time, which takes minutes. `--engine-cache` keeps the built engines for the next run.

```shell
carbon-ocr ocr scans/*.png --set models/fullerene-3M --device cuda --engine-cache .engines
carbon-ocr ocr scans/*.png --set models/fullerene-3M --device cuda --precision fp32
carbon-ocr serve --set models/fullerene-3M --device cuda --engine-cache .engines
carbon-ocr bench models/fullerene-3M/rec-latin.fp16.onnx --backend trt --shape x=64x1x32x640 --engine-cache .engines
```

## OCRmyPDF

`tools/ocrmypdf/carbon_ocrmypdf.py` makes Carbon-OCR the OCR engine of OCRmyPDF 17. OCRmyPDF turns and levels each page itself, and the plugin runs `carbon-ocr ocr --as-given` on it.

```shell
pip install -r tools/ocrmypdf/requirements.txt
ocrmypdf --plugin tools/ocrmypdf/carbon_ocrmypdf.py --carbon-set models/fullerene-3M in.pdf out.pdf
```